<a href="https://colab.research.google.com/github/IsaacFigNewton/Multiplexed-Hypergraph-Visualizer/blob/main/3D_Multiplexed_Hypergraph_Visual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download/Install dependencies

In [1]:
!pip install hypergraphx plotly scipy networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.3/140.3 kB 3.2 MB/s eta 0:00:00


# Test

In [2]:
import numpy as np
import networkx as nx
from scipy.spatial import ConvexHull
import plotly.graph_objects as go

from hypergraphx import Hypergraph
from hypergraphx.representations.projections import clique_projection

# --- Your data ---
multiplex_edges = [
    (frozenset({1, 2, 3, 4, 5, 6, 7, 8}), {"layer": 3}),
    (frozenset({8, 2, 4, 5}), {"layer": 2}),
    (frozenset({6, 7}), {"layer": 2}),
    (frozenset({3, 4, 5, 6, 7, 8}), {"layer": 1}),
    (frozenset({8, 4, 5}), {"layer": 0}),
    (frozenset({7}), {"layer": 0}),
]

layer_color = {
    0: "rgba(228,26,28,0.22)",
    1: "rgba(55,126,184,0.22)",
    2: "rgba(77,175,74,0.22)",
    3: "rgba(152,78,163,0.18)",
}
layer_line = {
    0: "rgb(228,26,28)",
    1: "rgb(55,126,184)",
    2: "rgb(77,175,74)",
    3: "rgb(152,78,163)",
}

# --- Group edges by layer; collect nodes ---
by_layer = {}
all_nodes = set()
layer_nodes = {}
for e, md in multiplex_edges:
    L = md["layer"]
    e = tuple(sorted(e))
    by_layer.setdefault(L, []).append(e)
    all_nodes |= set(e)
    layer_nodes.setdefault(L, set()).update(e)

layers = sorted(by_layer.keys())
minL, maxL = layers[0], layers[-1]

# --- Compute one global (x,y) layout for all nodes (so rods align) ---
H_all = Hypergraph([e for L in layers for e in by_layer[L]])
G_proj = clique_projection(H_all, keep_isolated=True)

G_layout = nx.Graph()
G_layout.add_nodes_from(sorted(all_nodes))
G_layout.add_edges_from(G_proj.edges())
pos = nx.spring_layout(G_layout, seed=10, iterations=250, k=0.7)  # dict node -> (x,y)

# --- Build 3D figure ---
fig = go.Figure()

# 1) Node rods (vertical lines through layers where the node exists)
for n in sorted(all_nodes):
    x, y = pos[n]
    # Rod spans the full layer range; if you prefer only layers where node appears, compute those.
    fig.add_trace(
        go.Scatter3d(
            x=[x, x], y=[y, y], z=[minL, maxL],
            mode="lines",
            line=dict(width=6, color="rgba(50,50,50,0.45)"),
            hoverinfo="text",
            text=[f"node {n}", f"node {n}"],
            showlegend=False,
        )
    )

# 2) Nodes as points on each layer plane (optional but helps readability)
for L in layers:
    xs, ys, zs, labels = [], [], [], []
    for n in sorted(layer_nodes.get(L, [])):
        x, y = pos[n]
        xs.append(x); ys.append(y); zs.append(L); labels.append(f"node {n} (layer {L})")
    fig.add_trace(
        go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode="markers+text",
            marker=dict(size=6, color="rgba(30,30,30,0.9)"),
            text=[str(n) for n in sorted(layer_nodes.get(L, []))],
            textposition="top center",
            hovertext=labels,
            showlegend=False,
        )
    )

# 3) Hyperedges on each layer: draw a filled polygon (convex hull) when size>=3
#    - size==2: draw a segment on that z-plane
#    - size==1: annotate (a marker) on that z-plane
for L in layers:
    for e in by_layer[L]:
        pts = np.array([pos[n] for n in e], dtype=float)  # shape (k,2)
        z = float(L)

        if len(e) == 1:
            n = e[0]
            x, y = pos[n]
            fig.add_trace(
                go.Scatter3d(
                    x=[x], y=[y], z=[z],
                    mode="markers",
                    marker=dict(size=10, color=layer_line[L]),
                    hoverinfo="text",
                    text=[f"hyperedge {{{n}}} (L{L})"],
                    showlegend=False,
                )
            )
        elif len(e) == 2:
            (u, v) = e
            x1, y1 = pos[u]; x2, y2 = pos[v]
            fig.add_trace(
                go.Scatter3d(
                    x=[x1, x2], y=[y1, y2], z=[z, z],
                    mode="lines",
                    line=dict(width=6, color=layer_line[L]),
                    hoverinfo="text",
                    text=[f"hyperedge {set(e)} (L{L})", f"hyperedge {set(e)} (L{L})"],
                    showlegend=False,
                )
            )
        else:
            # Convex hull in 2D
            hull = ConvexHull(pts)
            hull_pts = pts[hull.vertices]
            # close polygon
            hull_pts = np.vstack([hull_pts, hull_pts[0]])

            fig.add_trace(
                go.Scatter3d(
                    x=hull_pts[:, 0],
                    y=hull_pts[:, 1],
                    z=np.full(len(hull_pts), z),
                    mode="lines",
                    line=dict(width=6, color=layer_line[L]),
                    hoverinfo="text",
                    text=[f"hyperedge {set(e)} (L{L})"] * len(hull_pts),
                    showlegend=False,
                )
            )
            fig.add_trace(
                go.Mesh3d(
                    x=hull_pts[:-1, 0],
                    y=hull_pts[:-1, 1],
                    z=np.full(len(hull_pts) - 1, z),
                    alphahull=0,  # already convex; let plotly triangulate
                    color=layer_color[L],
                    opacity=0.35,
                    hoverinfo="skip",
                    showlegend=False,
                )
            )

# Layout polish
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(title="layer", dtick=1),
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    title="Multiplex hypergraph in 3D: nodes as rods across layers",
)
fig.show()